# Week 7 — Day 1: Sprint 2 Planning & Convolutional Neural Networks

**Project (image track):** Melanoma Skin Cancer Classification — Benign vs. Malignant
**Dataset:** [melanoma-skin-cancer-dataset-benign-vs-malignant](https://www.kaggle.com/datasets/ailearner-researchlab/melanoma-skin-cancer-dataset-benign-vs-malignant) (dermoscopic images, binary classification)

---

## Objective

Complete Sprint 2 planning, explain why dense networks fail on images and how convolution solves it, and confirm the CNN architecture as the right fit for this project's image data.


## Lesson Content

### 1.1 — Sprint 2 Planning

Sprint 2 opens with planning: define the Sprint 2 goal — **advancing the project's core model to beat the Week 6 tabular baseline** — and break it into backlog tasks, carrying forward the Sprint 1 retrospective action item (prioritize imbalance handling over architecture complexity).

**Sprint 2 Backlog:**
1. Load and inspect the melanoma image dataset; confirm class balance.
2. Apply a hand-defined convolution filter to a sample image and visualize the feature map.
3. Build a from-scratch CNN (Day 2).
4. Add data augmentation and transfer learning (Day 2).
5. (If the project also has sequential/text data) Build an LSTM baseline (Day 3).
6. Mentor Code & Notebook Review (Day 3).
7. Attention/Transformer exploration if applicable (Day 4).
8. Advance and tune the core model to decisively beat the baseline; Sprint Review & Retrospective (Day 5).

Since this project's data type is **images**, the architecture decision (per the program's own matrix) is clear: **CNN with transfer learning** is the appropriate core model — not the dense network from Week 6, and not an RNN/Transformer.

### 1.2 — Why Dense Networks Fail on Images

A modestly-sized 128×128 color image already has `128 × 128 × 3 = 49,152` numbers. Feeding that into a dense network the way Week 6 did means the first layer alone needs millions of weights — computationally hopeless, and it throws away a key fact: nearby pixels are related, and a pattern (a lesion border, a color transition) looks the same wherever it appears in the image. Dense networks discard this spatial structure entirely.

### 1.3 — Convolution: the Core Idea

A convolution slides a small filter (kernel) — e.g. a 3×3 grid of weights — across the image, computing a dot product at each position. This is still the Week 2 dot product, applied locally and repeatedly. Each filter learns to detect one specific pattern (an edge, a curve, a texture); the output is a **feature map** showing where that pattern appears.

| Concept | Meaning |
|---|---|
| Filter / kernel | A small grid of learnable weights that detects one pattern |
| Feature map | The output showing where the filter's pattern was found |
| Stride | How far the filter moves each step |
| Padding | Adding a border so the filter can process edge pixels |

### 1.4 — Why Convolution Wins

Two decisive advantages over dense layers:
- **Parameter sharing** — the same small filter is reused across the whole image, so a CNN needs vastly fewer weights than a dense network.
- **Translation invariance** — because the filter slides everywhere, a pattern (e.g. an irregular lesion border) is detected no matter where it appears in the image.

A CNN learns a **hierarchy**: early layers detect simple patterns (edges, color gradients), middle layers combine them (textures, shapes), and deep layers recognize whole structures (asymmetric lesions, irregular borders — the ABCDE features dermatologists look for). The network learns these features from data; they are not hand-designed.


---

## Hands-On Lab: Sprint 2 Kickoff & Convolution

1. Complete Sprint 2 planning (above) and select the core-model backlog tasks.
2. Load the melanoma dataset and confirm class balance.
3. On a sample image, apply a hand-defined edge-detection filter with a convolution and visualize the feature map.
4. Explain, in Markdown, why the same filter across the whole image needs far fewer weights than a dense layer.
5. Confirm the project's data type calls for a CNN, and record the decision.

**Tools:** TensorFlow/Keras, NumPy, Matplotlib, Jupyter/Colab, Git & GitHub


### Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

import tensorflow as tf

tf.random.set_seed(42)
np.random.seed(42)


### Step 1: Load the Dataset & Confirm Class Balance

Dataset structure (as distributed on Kaggle): a `train/` folder (and `test/`) with two subfolders, `Benign/` and `Malignant/`, each containing dermoscopic images.

If running in Colab, download via the Kaggle API or upload the dataset zip and extract it first — adjust `DATA_DIR` below to match your actual path.


In [ ]:
DATA_DIR = "melanoma_cancer_dataset"  # adjust to your actual extracted path
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

for split_dir, split_name in [(TRAIN_DIR, "Train"), (TEST_DIR, "Test")]:
    if os.path.isdir(split_dir):
        print(f"--- {split_name} ---")
        for cls in sorted(os.listdir(split_dir)):
            cls_path = os.path.join(split_dir, cls)
            if os.path.isdir(cls_path):
                n_images = len([f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
                print(f"  {cls}: {n_images} images")
    else:
        print(f"{split_name} directory not found at {split_dir} — update DATA_DIR.")


In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary", seed=42
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary", seed=42, shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)


In [ ]:
# Visualize a few sample images with their labels
plt.figure(figsize=(10, 5))
for images, labels in train_ds.take(1):
    for i in range(8):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i].numpy()[0])])
        plt.axis("off")
plt.tight_layout()
plt.show()


### Step 2: Hand-Defined Convolution — Edge Detection & Feature Map

Applying a classic Sobel-style vertical edge-detection filter to one sample image, computed manually (not via a trained CNN), to make the convolution operation from Section 1.3 concrete before building any real network.


In [ ]:
import tensorflow as tf

# Grab one sample image
for images, labels in train_ds.take(1):
    sample_image = images[0].numpy()
    sample_label = class_names[int(labels[0].numpy()[0])]
    break

gray = np.mean(sample_image, axis=-1)  # convert to grayscale for a clean edge-map visual

# Hand-defined vertical edge detection filter (Sobel-style, 3x3)
edge_filter = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
])

def convolve2d(image, kernel):
    kh, kw = kernel.shape
    h, w = image.shape
    out_h, out_w = h - kh + 1, w - kw + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            region = image[i:i+kh, j:j+kw]
            output[i, j] = np.sum(region * kernel)
    return output

feature_map = convolve2d(gray, edge_filter)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title(f"Original (grayscale) — {sample_label}")
axes[0].axis("off")

axes[1].imshow(feature_map, cmap="gray")
axes[1].set_title("Feature Map — Vertical Edge Filter")
axes[1].axis("off")

plt.tight_layout()
plt.show()


### Step 3: Why the Same Filter Needs Far Fewer Weights Than a Dense Layer

For a 128×128 grayscale image:
- A **dense layer** connecting every pixel to just 64 hidden neurons needs `128 × 128 × 64 = 1,048,576` weights.
- A **3×3 convolution filter** producing a comparable feature map needs only `3 × 3 = 9` weights — reused at every position in the image via sliding, not re-learned per location.

That's roughly **116,000× fewer parameters** for one filter versus one dense layer, because the filter's weights are **shared** across every spatial position rather than duplicated per pixel. This is exactly why CNNs scale to real image sizes while dense networks don't: the network isn't learning "what pattern is at pixel (14, 87)" and "what pattern is at pixel (14, 88)" separately — it learns one small, reusable pattern detector and slides it everywhere, which is also what gives it translation invariance (Section 1.4).


In [ ]:
dense_weights = 128 * 128 * 64
conv_weights = 3 * 3  # one 3x3 filter

print(f"Dense layer (128x128 -> 64 units): {dense_weights:,} weights")
print(f"One 3x3 convolution filter:         {conv_weights} weights")
print(f"Ratio: {dense_weights / conv_weights:,.0f}x fewer weights for the conv filter")


### Step 4: Architecture Decision — Recorded

**Project data type:** images (dermoscopic skin lesion photos, RGB, variable resolution).

**Architecture decision:** **CNN with transfer learning** (Day 2) is the correct core model for this project, per the program's data-type-to-architecture matrix. A dense network (Week 6) would be computationally impractical and would ignore the spatial structure that actually carries the diagnostic signal — lesion border irregularity, asymmetry, and color variation are inherently spatial patterns. An RNN/LSTM or Transformer is not applicable here since the data has no sequential/temporal structure.

**Next steps (Day 2):** build a from-scratch CNN, add data augmentation (the dataset's Kaggle page describes it as sizeable but benign/malignant photos still benefit from augmentation against overfitting), then apply transfer learning with a pre-trained model (e.g. MobileNetV2) — the most practical path to strong accuracy on medical image data with the compute available in Colab.
